# Chapter 7 — Regression Methods I: Simple & Multiple Linear Regression

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winnertechserv/aimlgybs/blob/main/part2_classic_ml/ch07_regression_methods_i/ch07_notebook.ipynb)

Companion notebook for **Chapter 7** of *AI-Machine Learning: Get Your Base Strong*. Read the chapter first, then run each cell and compare your output with the output printed in the book (shown below each cell).


In [ ]:
# Optional: install the exact library versions used for the outputs in the book.
# Google Colab already has these libraries; newer versions may change a few digits.
# !pip install -q numpy==2.5.3 pandas==3.0.5 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2 pillow==12.3.0

## Key takeaways

- Regression predicts a continuous output from one or more inputs.
- Simple linear regression fits a straight line, ŷ = w x + w₀. Multiple regression fits a plane or hyperplane, ŷ = wᵀx.
- Least squares picks the line that minimizes the sum of squared errors, and it has an exact closed-form answer.
- For many inputs, w = (XᵀX)⁻¹Xᵀy, using the pseudo-inverse of the data matrix.
- Each weight is the change in output for a one-unit change in that input, holding the others fixed.
- Judge a regression model on test data with squared error and RMSE.

## Example 1: Simple Linear Regression from Scratch

This program uses the exact closed-form formulas on the salaries of 10 employees. It finds the averages (9.0 years, 59.9 thousand), then the slope 3.04 and intercept 32.52. The table shows each training point, the line's prediction and the error. Finally it tests a new person with 10 years' experience: predicted 62.94, actual 64, squared error 1.12.

In [1]:
# Chapter 7 - Example 1: Simple linear regression from scratch (method of least squares)
# Salary (in $1000s) vs years of experience for 10 employees.
import numpy as np

x = np.array([1, 3, 4, 6, 7, 9, 11, 13, 16, 20], dtype=float)
y = np.array([35, 40, 47, 50, 56, 58, 67, 71, 82, 93], dtype=float)
N = len(x)

mu_x, mu_y = x.mean(), y.mean()
# Closed-form least-squares solution (set both derivatives of the error to zero)
w = np.sum((x - mu_x) * (y - mu_y)) / np.sum((x - mu_x) ** 2)
w0 = mu_y - w * mu_x
print(f"Mean of x = {mu_x:.1f}, mean of y = {mu_y:.1f}")
print(f"Slope w   = {w:.2f}   (each extra year adds about ${w * 1000:,.0f})")
print(f"Intercept w0 = {w0:.2f}  (predicted starting salary)")

# Training error
y_hat = w * x + w0
E = 0.5 * np.sum((y_hat - y) ** 2)
print(f"\nSum-of-squares error on training data, E = {E:.2f}")
print("  x    y   predicted  error")
for xi, yi, pi in zip(x, y, y_hat):
    print(f" {xi:3.0f}  {yi:3.0f}   {pi:7.2f}  {pi - yi:6.2f}")

# Test phase: a new person with 10 years of experience earns 64 (thousand)
x_test, y_test = 10, 64.0
pred = w * x_test + w0
print(f"\nTest: 10 years -> predicted {pred:.2f}, actual {y_test:.0f}, "
      f"squared error {(pred - y_test) ** 2:.2f}")

Mean of x = 9.0, mean of y = 59.9
Slope w   = 3.04   (each extra year adds about $3,043)
Intercept w0 = 32.52  (predicted starting salary)

Sum-of-squares error on training data, E = 10.15
  x    y   predicted  error
   1   35     35.56    0.56
   3   40     41.64    1.64
   4   47     44.69   -2.31
   6   50     50.77    0.77
   7   56     53.81   -2.19
   9   58     59.90    1.90
  11   67     65.99   -1.01
  13   71     72.07    1.07
  16   82     81.20   -0.80
  20   93     93.37    0.37

Test: 10 years -> predicted 62.94, actual 64, squared error 1.12


## Example 2: Multiple Linear Regression with the Pseudo-Inverse

Two inputs (humidity and pressure) predict temperature. We build the data matrix with a column of 1s and solve w = (XᵀX)⁻¹Xᵀy. The normal equation and NumPy's pseudo-inverse give the same weights: [178.60, −0.2754, −0.1349]. The test prediction is 22.84 °C against an actual 21.7 °C. Across the training days the RMSE is 1.21 °C, under 5% of the average temperature.

In [2]:
# Chapter 7 - Example 2: Multiple linear regression with the pseudo-inverse
# Predict temperature (C) from humidity (%) and air pressure (hPa) on 15 days.
import numpy as np
np.set_printoptions(suppress=True)

data = np.array([
    [45.3, 1012.7, 29.3], [46.0, 1017.9, 29.3], [53.7, 1021.4, 25.2],
    [62.7, 1023.2, 22.0], [81.8, 1023.1, 18.8], [59.8, 1021.6, 23.4],
    [57.2, 1018.5, 23.7], [82.4, 1022.0, 18.6], [81.5, 1028.3, 18.7],
    [43.5, 1011.0, 32.0], [48.4, 1024.5, 27.1], [68.5, 1027.9, 21.3],
    [55.5, 1014.5, 24.1], [42.9, 1016.8, 31.6], [42.8, 1012.3, 30.6]])
X_raw, y = data[:, :2], data[:, 2]

# Data matrix: a column of 1s (for w0) followed by the inputs
X = np.column_stack([np.ones(len(X_raw)), X_raw])
print("Data matrix X has shape", X.shape, "(N rows, d+1 columns)")

w = np.linalg.inv(X.T @ X) @ X.T @ y        # w = (X^T X)^-1 X^T y
w_pinv = np.linalg.pinv(X) @ y              # same thing, numerically safer
print("w (normal equation) =", np.round(w, 4))
print("w (pseudo-inverse)  =", np.round(w_pinv, 4))

x_test = np.array([1, 66.0, 1020.0])
pred = x_test @ w
print(f"\nTest: humidity 66.0, pressure 1020.0 -> predicted {pred:.2f} C "
      f"(actual 21.7, squared error {(pred - 21.7) ** 2:.4f})")

y_hat = X @ w
rmse = np.sqrt(np.mean((y_hat - y) ** 2))
print(f"Training RMSE = {rmse:.3f} C  ({100 * rmse / y.mean():.1f}% of the mean temperature)")
print("Meaning of the weights: each extra 1% humidity changes temperature by "
      f"{w[1]:+.3f} C, holding pressure fixed.")

Data matrix X has shape (15, 3) (N rows, d+1 columns)
w (normal equation) = [178.5985  -0.2754  -0.1349]
w (pseudo-inverse)  = [178.5985  -0.2754  -0.1349]

Test: humidity 66.0, pressure 1020.0 -> predicted 22.84 C (actual 21.7, squared error 1.3024)
Training RMSE = 1.212 C  (4.8% of the mean temperature)
Meaning of the weights: each extra 1% humidity changes temperature by -0.275 C, holding pressure fixed.


## Example 3: Closed Form vs Gradient Descent on Real Data

Scikit-learn's diabetes dataset has 442 patients and 10 measurements each. We fit linear regression two ways. The closed form gives the answer in one step. Gradient descent starts from zero and walks downhill: watch the training error fall from 23,375 to 2,804. Both reach the same test RMSE (about 55.7), and so does scikit-learn. We standardize the features first because gradient descent converges much faster when all inputs are on the same scale.

In [3]:
# Chapter 7 - Example 3: Closed form vs gradient descent, judged on unseen test data
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

X, y = load_diabetes(return_X_y=True)        # 442 patients, 10 measurements each
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)
scaler = StandardScaler().fit(X_tr)           # same scale for every feature helps GD
X_tr, X_te = scaler.transform(X_tr), scaler.transform(X_te)
add1 = lambda A: np.column_stack([np.ones(len(A)), A])

# 1. Closed form (normal equation via the pseudo-inverse)
w_cf = np.linalg.pinv(add1(X_tr)) @ y_tr

# 2. Gradient descent: walk downhill on the error surface step by step
A = add1(X_tr)
w_gd = np.zeros(A.shape[1])
lr = 0.1
for epoch in range(1, 3001):
    grad = A.T @ (A @ w_gd - y_tr) / len(y_tr)
    w_gd -= lr * grad
    if epoch in (1, 10, 100, 1000, 3000):
        mse = np.mean((A @ w_gd - y_tr) ** 2)
        print(f"GD epoch {epoch:5d}: training MSE = {mse:9.1f}")

# 3. scikit-learn for comparison
sk = LinearRegression().fit(X_tr, y_tr)

def rmse(pred): return np.sqrt(np.mean((pred - y_te) ** 2))
print(f"\nTest RMSE  closed form     : {rmse(add1(X_te) @ w_cf):.2f}")
print(f"Test RMSE  gradient descent: {rmse(add1(X_te) @ w_gd):.2f}")
print(f"Test RMSE  scikit-learn    : {rmse(sk.predict(X_te)):.2f}")
print(f"Largest weight difference, closed form vs GD: {np.abs(w_cf - w_gd).max():.3f}")
print(f"Test RMSE as % of the mean target: {100 * rmse(add1(X_te) @ w_cf) / y_te.mean():.1f}%")

GD epoch     1: training MSE =   23374.7
GD epoch    10: training MSE =    5816.5
GD epoch   100: training MSE =    2812.7
GD epoch  1000: training MSE =    2805.7
GD epoch  3000: training MSE =    2804.2

Test RMSE  closed form     : 55.65
Test RMSE  gradient descent: 55.67
Test RMSE  scikit-learn    : 55.65
Largest weight difference, closed form vs GD: 1.550
Test RMSE as % of the mean target: 36.6%
